# Milestone 2: Baseline Generation and Parameter Experiments

| Exp. | Script | Setting varied | Images |
|---|---|---|---|
| 1 | `01_baseline.py` | none (SD v1.5, DDIM, 30 steps, w = 7.5), 200 COCO captions | 200 |
| 2 | `02_guidance_sweep.py` | guidance w ∈ {1, 3, 5, 7.5, 10, 15}, 32 captions × 2 seeds | 384 |
| 3 | `03_scheduler_sweep.py` | DDIM, DDPM, PNDM, Euler, Euler-a, DPM++ × {10, 25, 50} steps, 24 captions | 432 |
| 4 | `04_noise_schedule.py` | Euler with leading, linspace, trailing, Karras spacing × {10, 25} steps, plus a 50-step reference, 24 captions | 216 |

Captions come from the Milestone 1 pair manifest. Caption *i* uses seed *i* in every experiment. Each image is scored with CLIP score (ViT-L/14) against its caption.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IE7615_Project2'
except Exception as e:
    print('Google Drive not mounted:', e)
    DRIVE = '/content/outputs'
REPO = 'https://github.com/anshuman-dataverse/text-to-image-generation.git'
os.makedirs(f'{DRIVE}/milestone2_outputs', exist_ok=True)
os.makedirs(f'{DRIVE}/data', exist_ok=True)

In [ ]:
%cd /content
!test -d project || git clone -q $REPO project
!cd project && git remote set-url origin $REPO && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
!pip -q install -r project/requirements.txt
!pip -q uninstall -y torchao
!rm -rf project/outputs/milestone2/images && ln -sfn "$DRIVE/milestone2_outputs" project/outputs/milestone2/images
!rm -rf project/data && ln -sfn "$DRIVE/data" project/data
!test -d project/data/processed/images && echo "COCO images found" || echo "COCO images not found

In [ ]:
%cd /content/project

import json, subprocess
from pathlib import Path
from IPython.display import Image, display

SCRIPTS = Path('scripts/milestone2')
OUT = Path('outputs/milestone2')

def run(script, *args):
    log = OUT / 'logs' / (Path(script).stem + '.log')
    log.parent.mkdir(parents=True, exist_ok=True)
    commit = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
    print('code version:', commit)
    with open(log, 'w') as f:
        f.write(f'code version: {commit}\n')
        p = subprocess.Popen(['python', '-u', str(SCRIPTS / script), *map(str, args)], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            f.write(line)
    if p.wait():
        raise RuntimeError(f'{script} failed (exit code {p.returncode})')

def show(*names):
    for n in names:
        path = OUT / 'figures' / n
        display(Image(str(path))) if path.exists() else print(f'missing: {path}')

## 1. Baseline over COCO captions (DDIM, 30 steps, w = 7.5)

The Milestone 1 settings on 200 COCO captions. CLIP score = 100 · max(cos(image, caption), 0) with CLIP ViT-L/14. The real COCO photo is scored against the same caption for comparison.

In [ ]:
run('01_baseline.py')

In [ ]:
show('baseline_samples.png', 'baseline_gallery.png', 'baseline_clip_hist.png', 'baseline_clip_extremes.png')

## 2. Classifier-free guidance scale

The same 32 captions at w ∈ {1, 3, 5, 7.5, 10, 15}, two seeds per caption. Seed diversity is 1 − cosine similarity between the CLIP embeddings of the two images. At w = 1 there is no guidance.

In [ ]:
run('02_guidance_sweep.py')

In [ ]:
for s in json.load(open(OUT / 'results/guidance_sweep_summary.json'))['per_scale']:
    print(f"w={s['guidance']:>4}  CLIP {s['clip_score']:.2f} ± {s['clip_score_ci95']:.2f}  "
          f"diversity {s.get('diversity', float('nan')):.3f}  saturation {s['saturation']:.3f}  {s['seconds_per_image']} s/img")
show('guidance_curves.png', 'guidance_grid.png')

## 3. Scheduler × sampling steps

Six schedulers at 10, 25 and 50 steps with w = 7.5. Each image is also compared with the same scheduler's 50-step image.

In [ ]:
run('03_scheduler_sweep.py')

In [ ]:
for s in json.load(open(OUT / 'results/scheduler_sweep_summary.json'))['per_config']:
    print(f"{s['scheduler']:>8} {s['steps']:>3} steps  CLIP {s['clip_score']:.2f} ± {s['clip_score_ci95']:.2f}  "
          f"Δpixel→50 {s['pixel_diff_to_50']}  {s['seconds_per_image']} s/img")
show('scheduler_heatmaps.png', 'scheduler_clip_vs_steps.png', 'scheduler_grid.png')

## 4. Noise schedule (Euler, w = 7.5)

The sampler is fixed and only the spacing of the noise levels changes: leading (SD v1.5 default), linspace, trailing and Karras, at 10 and 25 steps, compared with leading at 50 steps.

In [ ]:
run('04_noise_schedule.py')

In [ ]:
for s in json.load(open(OUT / 'results/noise_schedule_summary.json'))['per_config']:
    print(f"{s['schedule']:>8} {s['steps']:>3} steps  CLIP {s['clip_score']:.2f} ± {s['clip_score_ci95']:.2f}  "
          f"Δpixel→ref {s['pixel_diff_to_ref']}  first σ {s['first_sigma']}  {s['seconds_per_image']} s/img")
show('noise_schedule_sigmas.png', 'noise_schedule_metrics.png', 'noise_schedule_grid.png')

## 5. Save outputs

In [ ]:
%cd /content/project
!rm -f "$DRIVE/milestone2_outputs.zip"
!zip -qr "$DRIVE/milestone2_outputs.zip" outputs/milestone2/results outputs/milestone2/figures outputs/milestone2/logs
!ls -la "$DRIVE"